# HDBSCAN Parameter Persistence Analysis

Compute cluster persistence scores across parameter choices to identify robust structures.

In [ ]:
import numpy as np
import h5py
import re
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("../output")
OVERLAP_THRESHOLD = 0.5  # Minimum member overlap to consider clusters "the same"

## 1. Load All Parameter Runs

In [ ]:
def parse_filename(filename):
    """Extract (min_cluster_size, min_samples) from filename."""
    match = re.search(r'mcs_(\d+)_ms_(\d+)', filename)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None

def load_cluster_members(filepath):
    """Load cluster member indices from HDF5 file."""
    clusters = {}
    with h5py.File(filepath, 'r') as f:
        if 'clusters' not in f:
            return clusters
        for cluster_name in f['clusters'].keys():
            cluster_id = int(cluster_name.split('_')[1])
            grp = f[f'clusters/{cluster_name}']
            # Get member realization IDs and halo indices as unique identifiers
            if 'members' in grp and 'realization_id' in grp['members']:
                real_ids = grp['members/realization_id'][:]
                halo_ids = grp['members/halo_index'][:] if 'halo_index' in grp['members'] else np.arange(len(real_ids))
                # Create unique member identifiers
                members = set(zip(real_ids, halo_ids))
            else:
                # Fallback: use member indices from assignments
                member_indices = grp['member_indices'][:] if 'member_indices' in grp else np.array([])
                members = set(member_indices)
            
            # Also store summary stats for reporting
            clusters[cluster_id] = {
                'members': members,
                'n_members': len(members),
                'existence_prob': grp.attrs.get('existence_prob', np.nan),
                'center_xyz': grp.attrs.get('center_xyz', np.array([np.nan]*3)),
                'mean_m200_mass': grp.attrs.get('mean_m200_mass', np.nan)
            }
    return clusters

# Find all HDBSCAN output files
files = sorted(OUTPUT_DIR.glob("hdbscan_clusters_mcs_*_ms_*.h5"))
print(f"Found {len(files)} parameter runs:")

runs = {}
for f in files:
    mcs, ms = parse_filename(f.name)
    if mcs is not None:
        runs[(mcs, ms)] = load_cluster_members(f)
        print(f"  mcs={mcs}, ms={ms}: {len(runs[(mcs, ms)])} clusters")

if len(runs) < 2:
    raise ValueError("Need at least 2 parameter runs for persistence analysis")

## 2. Match Clusters Across Runs

In [ ]:
def compute_overlap(members_a, members_b):
    """Compute Jaccard-like overlap between two member sets."""
    if len(members_a) == 0 or len(members_b) == 0:
        return 0.0
    intersection = len(members_a & members_b)
    # Use minimum size for asymmetric matching (does A appear in B?)
    return intersection / min(len(members_a), len(members_b))

def find_matching_cluster(target_members, candidate_clusters, threshold=OVERLAP_THRESHOLD):
    """Find best matching cluster in candidates, if above threshold."""
    best_overlap = 0
    best_id = None
    for cid, cdata in candidate_clusters.items():
        overlap = compute_overlap(target_members, cdata['members'])
        if overlap > best_overlap:
            best_overlap = overlap
            best_id = cid
    if best_overlap >= threshold:
        return best_id, best_overlap
    return None, 0

# Build unified cluster catalog by matching across all runs
# Use the run with most clusters as reference, then match others to it
param_list = list(runs.keys())
reference_params = max(param_list, key=lambda p: len(runs[p]))
print(f"Reference run: mcs={reference_params[0]}, ms={reference_params[1]} ({len(runs[reference_params])} clusters)")

# For each reference cluster, track which runs it appears in
persistence_data = {}
for ref_cid, ref_data in runs[reference_params].items():
    appearances = {reference_params: ref_cid}  # Always appears in reference
    
    for params in param_list:
        if params == reference_params:
            continue
        match_id, overlap = find_matching_cluster(ref_data['members'], runs[params])
        if match_id is not None:
            appearances[params] = match_id
    
    persistence_data[ref_cid] = {
        'appearances': appearances,
        'persistence': len(appearances) / len(runs),
        'n_members': ref_data['n_members'],
        'existence_prob': ref_data['existence_prob'],
        'center_xyz': ref_data['center_xyz'],
        'mean_m200_mass': ref_data['mean_m200_mass']
    }

print(f"\nMatched {len(persistence_data)} clusters from reference run")

## 3. Persistence Score Distribution

In [ ]:
persistence_scores = np.array([d['persistence'] for d in persistence_data.values()])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(persistence_scores, bins=len(runs), edgecolor='black', alpha=0.7)
axes[0].axvline(1.0, color='green', linestyle='--', linewidth=2, label='Perfect (1.0)')
axes[0].axvline(0.7, color='orange', linestyle='--', linewidth=2, label='Threshold (0.7)')
axes[0].set_xlabel('Persistence Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Cluster Persistence Distribution')
axes[0].legend()

# Persistence vs existence probability
exist_probs = np.array([d['existence_prob'] for d in persistence_data.values()])
axes[1].scatter(exist_probs, persistence_scores, alpha=0.6, s=40)
axes[1].set_xlabel('Existence Probability')
axes[1].set_ylabel('Persistence Score')
axes[1].set_title('Persistence vs Existence Probability')
axes[1].axhline(0.7, color='orange', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Summary stats
n_perfect = np.sum(persistence_scores == 1.0)
n_robust = np.sum(persistence_scores >= 0.7)
n_fragile = np.sum(persistence_scores < 0.5)

print(f"\nPersistence Summary:")
print(f"  Perfect (1.0): {n_perfect} clusters")
print(f"  Robust (≥0.7): {n_robust} clusters")
print(f"  Fragile (<0.5): {n_fragile} clusters")

## 4. Optimal Parameter Recommendation

In [ ]:
# For each parameter setting, count how many robust clusters it recovers
param_scores = {}

for params in param_list:
    n_robust_recovered = 0
    n_total_recovered = 0
    
    for ref_cid, pdata in persistence_data.items():
        if params in pdata['appearances']:
            n_total_recovered += 1
            if pdata['persistence'] >= 0.7:
                n_robust_recovered += 1
    
    param_scores[params] = {
        'n_clusters': len(runs[params]),
        'n_robust_recovered': n_robust_recovered,
        'n_total_recovered': n_total_recovered,
        'robust_fraction': n_robust_recovered / len(runs[params]) if len(runs[params]) > 0 else 0
    }

print("Parameter Comparison:")
print("=" * 70)
print(f"{'mcs':>5} {'ms':>5} {'clusters':>10} {'robust':>10} {'robust_frac':>12}")
print("-" * 70)

for params in sorted(param_scores.keys()):
    s = param_scores[params]
    print(f"{params[0]:>5} {params[1]:>5} {s['n_clusters']:>10} {s['n_robust_recovered']:>10} {s['robust_fraction']:>12.1%}")

# Recommend: highest robust fraction, then most clusters as tiebreaker
best_params = max(param_scores.keys(), 
                  key=lambda p: (param_scores[p]['robust_fraction'], param_scores[p]['n_clusters']))

print(f"\n** RECOMMENDED: mcs={best_params[0]}, ms={best_params[1]} **")
print(f"   {param_scores[best_params]['n_clusters']} clusters, "
      f"{param_scores[best_params]['robust_fraction']:.1%} robust")

## 5. Export Persistence Scores for Final Catalog

In [ ]:
# Choose final parameters (use recommended or override)
FINAL_PARAMS = best_params  # or set manually, e.g., (12, 12)

# Build persistence lookup for final catalog
final_persistence = {}

for ref_cid, pdata in persistence_data.items():
    if FINAL_PARAMS in pdata['appearances']:
        final_cid = pdata['appearances'][FINAL_PARAMS]
        final_persistence[final_cid] = pdata['persistence']

# Also check for clusters in final run not in reference (assign based on their own matching)
for final_cid, final_data in runs[FINAL_PARAMS].items():
    if final_cid not in final_persistence:
        # This cluster wasn't in reference - compute its persistence directly
        appearances = 1  # appears in final
        for params in param_list:
            if params == FINAL_PARAMS:
                continue
            match_id, _ = find_matching_cluster(final_data['members'], runs[params])
            if match_id is not None:
                appearances += 1
        final_persistence[final_cid] = appearances / len(runs)

print(f"Final catalog: mcs={FINAL_PARAMS[0]}, ms={FINAL_PARAMS[1]}")
print(f"Clusters with persistence scores: {len(final_persistence)}")
print(f"\nTop 10 most robust clusters:")

for cid in sorted(final_persistence.keys(), key=lambda x: -final_persistence[x])[:10]:
    cdata = runs[FINAL_PARAMS][cid]
    print(f"  Cluster {cid}: persistence={final_persistence[cid]:.2f}, "
          f"exist_prob={cdata['existence_prob']:.2f}, mass={cdata['mean_m200_mass']:.2e}")

In [ ]:
# Save persistence scores to the final HDF5 file
final_file = OUTPUT_DIR / f"hdbscan_clusters_mcs_{FINAL_PARAMS[0]}_ms_{FINAL_PARAMS[1]}.h5"

with h5py.File(final_file, 'r+') as f:
    # Add persistence scores to each cluster
    for cid, persistence in final_persistence.items():
        cluster_path = f'clusters/cluster_{cid}'
        if cluster_path in f:
            f[cluster_path].attrs['persistence_score'] = persistence
    
    # Add persistence metadata
    if 'persistence_analysis' in f:
        del f['persistence_analysis']
    pa = f.create_group('persistence_analysis')
    pa.attrs['n_parameter_runs'] = len(runs)
    pa.attrs['parameter_grid'] = str(list(runs.keys()))
    pa.attrs['overlap_threshold'] = OVERLAP_THRESHOLD
    pa.attrs['reference_params'] = str(reference_params)

print(f"Saved persistence scores to {final_file}")

## Summary

Each cluster now has a **persistence score** (0-1) indicating robustness across parameter choices:
- **1.0**: Appears in all parameter runs - definitely real
- **≥0.7**: Robust - appears in most runs
- **<0.5**: Fragile - parameter-dependent, treat with caution

Use `persistence_score` alongside `existence_prob` for filtering:
```python
# High-confidence clusters
robust = (persistence_score >= 0.7) & (existence_prob >= 0.5)
```